# 03 · Treinamento dos Modelos

Treinamos todos os modelos e salvamos os artefatos em `models/`.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib, json

from src.models.train import (
    compute_scale_pos_weight, apply_smote,
    train_logistic_regression, train_random_forest,
    train_xgboost, train_lightgbm, train_catboost,
    train_isolation_forest,
)
from src.models.evaluate import evaluate_model, evaluate_isolation_forest, summary_table
from src.visualization.plots import plot_roc_pr_curves, plot_confusion_matrices


## 1. Carrega Dados Processados


In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')


## 2. Prepara Balanceamento


In [ ]:
spw = compute_scale_pos_weight(y_train)
X_res, y_res = apply_smote(X_train, y_train)


## 3. Treina os Modelos


In [ ]:
lr  = train_logistic_regression(X_train, y_train)
rf  = train_random_forest(X_res, y_res)
xgb = train_xgboost(X_train, y_train, X_test, y_test, spw)
lgb = train_lightgbm(X_train, y_train, X_test, y_test, spw)
cat = train_catboost(X_train, y_train, spw)
iso = train_isolation_forest(X_train, y_train)


## 4. Avalia os Modelos


In [ ]:
results = [
    evaluate_model(lr,  X_test, y_test, 'Logistic Regression'),
    evaluate_model(rf,  X_test, y_test, 'Random Forest'),
    evaluate_model(xgb, X_test, y_test, 'XGBoost'),
    evaluate_model(lgb, X_test, y_test, 'LightGBM'),
    evaluate_model(cat, X_test, y_test, 'CatBoost'),
    evaluate_isolation_forest(iso, X_test, y_test),
]

# Adiciona y_test em cada resultado para as funções de plot
for r in results:
    r['y_test'] = y_test


In [ ]:
print('\n=== Tabela Comparativa ===')
summary_table(results)


In [ ]:
plot_roc_pr_curves(results)
plot_confusion_matrices(results)


## 5. Salva Artefatos


In [ ]:
os.makedirs('../models', exist_ok=True)

# Salva o melhor modelo (XGBoost) e os metadados necessários pela API
joblib.dump(xgb,           '../models/modelo_fraude.pkl')
joblib.dump(feature_names, '../models/features.pkl')

# Salva todos os modelos para comparação futura
for nome, modelo in [('lr', lr), ('rf', rf), ('xgb', xgb),
                       ('lgb', lgb), ('cat', cat), ('iso', iso)]:
    joblib.dump(modelo, f'../models/{nome}.pkl')

print('Todos os modelos salvos em models/ ✅')
